In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, confusion_matrix, roc_curve,
                             matthews_corrcoef)

In [ ]:
def optimal_threshold(t, p):
    fpr, tpr, thresholds = roc_curve(t, p)
    youden_j = tpr - fpr
    return thresholds[np.argmax(youden_j)]

def print_metrics(all_targets, all_preds, disease_cols, disease_meta, threshold_method='youden'):

    if threshold_method == 'youden':
        print("Threshold: Youden's J (maximises sensitivity + specificity)")
    else:
        print("Threshold: 0.5 (standard literature default)")

    print(f"\n{'Disease':30s}  {'PRAUC':>6}  {'AUROC':>6}  {'Sens':>6}  {'Spec':>6}  {'F1':>6}  {'MCC':>6}  {'Thresh':>6}  {'Prev':>6}")
    print("-" * 110)

    for k, col in enumerate(disease_cols):
        valid = ~np.isnan(all_targets[:, k])
        if valid.sum() == 0:
            continue

        t = all_targets[valid, k]
        p = all_preds[valid, k]
        prev = t.mean()

        if threshold_method == 'youden':
            thresh = optimal_threshold(t, p)
        else:
            thresh = 0.5

        pred_binary = (p >= thresh).astype(int)

        prauc = average_precision_score(t, p)
        auroc = roc_auc_score(t, p)
        f1    = f1_score(t, pred_binary, zero_division=0)
        mcc   = matthews_corrcoef(t, pred_binary)

        tn, fp, fn, tp = confusion_matrix(t, pred_binary, labels=[0,1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        print(f"  {disease_meta[col]:30s}  {prauc:6.3f}  {auroc:6.3f}  {sensitivity:6.3f}  {specificity:6.3f}  {f1:6.3f}  {mcc:6.3f}  {thresh:6.3f}  {prev:6.3f}")

    print("-" * 110)


# ── Run both ──────────────────────────────────────────────────────────────────
print_metrics(all_targets, all_preds, disease_cols, disease_meta, threshold_method='youden')
print("\n")
print_metrics(all_targets, all_preds, disease_cols, disease_meta, threshold_method='fixed')

In [ ]:
def head_feature_importance(model, loader, demo_cols, disease_cols, device, n_repeats=5):
    """
    For each disease head, permute each feature and measure drop in AUROC.
    Larger drop = more important feature.
    """
    from sklearn.metrics import roc_auc_score

    model.eval()

    # Get baseline predictions
    all_X, all_D = [], []
    with torch.no_grad():
        for X_batch, D_batch in loader:
            all_X.append(X_batch)
            all_D.append(D_batch)
    all_X = torch.cat(all_X).to(device)
    all_D = torch.cat(all_D).to(device)

    with torch.no_grad():
        base_preds = model(all_X, all_D).cpu().numpy()
    targets = all_D.cpu().numpy()

    # Per disease head, per feature — permute and measure drop
    results = {}
    for k, col in enumerate(disease_cols):
        valid = ~np.isnan(targets[:, k])
        t = targets[valid, k]
        base_auroc = roc_auc_score(t, base_preds[valid, k])

        importances = []
        for j, feat in enumerate(demo_cols):
            drops = []
            for _ in range(n_repeats):
                X_perm = all_X.clone()
                idx = torch.randperm(X_perm.shape[0])
                X_perm[:, j] = X_perm[idx, j]  # shuffle feature j

                with torch.no_grad():
                    perm_preds = model(X_perm, all_D).cpu().numpy()

                perm_auroc = roc_auc_score(t, perm_preds[valid, k])
                drops.append(base_auroc - perm_auroc)
            importances.append(np.mean(drops))

        results[col] = dict(zip(demo_cols, importances))

    return results, base_preds, targets

importance, _, _ = head_feature_importance(model, val_loader, demo_cols, disease_cols, device)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def head_feature_importance(model, loader, demo_cols, disease_cols, device, n_repeats=5):
    
    all_features = demo_cols + disease_cols

    model.eval()
    all_X, all_D = [], []
    with torch.no_grad():
        for X_batch, D_batch in loader:
            all_X.append(X_batch)
            all_D.append(D_batch)
    all_X = torch.cat(all_X).to(device)
    all_D = torch.cat(all_D).to(device)

    with torch.no_grad():
        base_preds = model(all_X, all_D).cpu().numpy()
    targets = all_D.cpu().numpy()

    results = {}
    for k, col in enumerate(disease_cols):
        valid = ~np.isnan(targets[:, k])
        t = targets[valid, k]
        base_auroc = roc_auc_score(t, base_preds[valid, k])

        importances = []
        for feat in all_features:
            drops = []
            for _ in range(n_repeats):
                X_perm = all_X.clone()
                D_perm = all_D.clone()
                idx = torch.randperm(all_X.shape[0])

                if feat in demo_cols:
                    j_demo = demo_cols.index(feat)
                    X_perm[:, j_demo] = X_perm[idx, j_demo]
                else:
                    j_dis = disease_cols.index(feat)
                    if j_dis != k:
                        D_perm[:, j_dis] = D_perm[idx, j_dis]

                with torch.no_grad():
                    perm_preds = model(X_perm, D_perm).cpu().numpy()

                perm_auroc = roc_auc_score(t, perm_preds[valid, k])
                drops.append(base_auroc - perm_auroc)
            importances.append(np.mean(drops))

        results[col] = dict(zip(all_features, importances))

    return results, base_preds, targets


def get_original_range(feat, demo_cols, disease_cols, continuous_cols, scaler, all_X_val, all_D_val, n_grid=50):
    if feat in demo_cols:
        j = demo_cols.index(feat)
        feat_min = all_X_val[:, j].min().item()
        feat_max = all_X_val[:, j].max().item()
        grid_scaled = torch.linspace(feat_min, feat_max, n_grid)

        if feat in continuous_cols:
            scaler_idx = continuous_cols.index(feat)
            dummy = np.zeros((n_grid, len(continuous_cols)))
            dummy[:, scaler_idx] = grid_scaled.numpy()
            original = scaler.inverse_transform(dummy)[:, scaler_idx]
        else:
            original = grid_scaled.numpy()

        return grid_scaled, original, 'demo'

    else:
        grid_scaled = torch.linspace(0.0, 1.0, n_grid)
        return grid_scaled, grid_scaled.numpy(), 'disease'


def plot_pdps(model, all_X, all_D, demo_cols, disease_cols, disease_meta, demo_meta,
              disease_col, importance_dict, device, continuous_cols, scaler,
              n_top=12, n_grid=50):

    model.eval()
    k = disease_cols.index(disease_col)

    sorted_feats = sorted(importance_dict[disease_col].items(),
                          key=lambda x: x[1], reverse=True)[:n_top]

    n_cols = 4
    n_rows = int(np.ceil(n_top / n_cols))   # ← dynamic grid: 12 → 3 rows × 4 cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))
    axes = axes.flat  # flatten for easy iteration

    fig.suptitle(f'Partial Dependence Plots — {disease_meta[disease_col]}',
                 fontsize=14, fontweight='bold', y=1.01)

    for ax, (feat, imp) in zip(axes, sorted_feats):
        grid_scaled, grid_original, feat_type = get_original_range(
            feat, demo_cols, disease_cols, continuous_cols, scaler, all_X, all_D, n_grid
        )

        mean_preds = []
        with torch.no_grad():
            for val in grid_scaled:
                X_pdp = all_X.clone()
                D_pdp = all_D.clone()

                if feat_type == 'demo':
                    j = demo_cols.index(feat)
                    X_pdp[:, j] = val
                else:
                    j = disease_cols.index(feat)
                    D_pdp[:, j] = val

                preds = model(X_pdp, D_pdp)
                mean_preds.append(preds[:, k].mean().item())

        if feat in disease_cols:
            label = disease_meta.get(feat, feat)
            color = '#1a5276'
        else:
            label = demo_meta.get(feat, feat)
            color = '#2c7a4b'

        ax.plot(grid_original, mean_preds, color=color, linewidth=2.5)
        ax.fill_between(grid_original, mean_preds, alpha=0.12, color=color)
        ax.set_xlabel(label, fontsize=8)
        ax.set_ylabel('Mean P(disease)', fontsize=8)
        ax.set_title(f'{label}\n(importance={imp:.4f})', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)

    # Hide any unused axes
    for ax in list(axes)[len(sorted_feats):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────
importance, base_preds, targets = head_feature_importance(
    model, val_loader, demo_cols, disease_cols, device
)

all_X_val, all_D_val = [], []
with torch.no_grad():
    for X_batch, D_batch in val_loader:
        all_X_val.append(X_batch)
        all_D_val.append(D_batch)
all_X_val = torch.cat(all_X_val).to(device)
all_D_val = torch.cat(all_D_val).to(device)

plot_pdps(model, all_X_val, all_D_val, demo_cols, disease_cols, disease_meta, demo_meta,
          disease_col='MCQ220', importance_dict=importance, device=device,
          continuous_cols=continuous_cols, scaler=scaler)

In [ ]:
# ── Disease-to-Disease Importance Heatmap ─────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Extract only disease-to-disease importances from the full importance dict
disease_importance_matrix = np.zeros((len(disease_cols), len(disease_cols)))

for k, target_col in enumerate(disease_cols):
    for j, input_col in enumerate(disease_cols):
        if input_col in importance[target_col]:
            disease_importance_matrix[k, j] = importance[target_col][input_col]

# Mask diagonal (disease can't predict itself)
np.fill_diagonal(disease_importance_matrix, np.nan)

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 11))

labels = [disease_meta[col] for col in disease_cols]

im = ax.imshow(disease_importance_matrix, cmap='YlOrRd', aspect='auto')

# Axis labels
ax.set_xticks(range(len(disease_cols)))
ax.set_yticks(range(len(disease_cols)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

ax.set_xlabel('Input disease (driver)', fontsize=11, labelpad=10)
ax.set_ylabel('Target disease (being predicted)', fontsize=11, labelpad=10)
ax.set_title('Disease-to-Disease Permutation Importance\n(how much each input disease drives prediction of each target)',
             fontsize=12, fontweight='bold', pad=15)

# Add value annotations
for i in range(len(disease_cols)):
    for j in range(len(disease_cols)):
        if i != j and not np.isnan(disease_importance_matrix[i, j]):
            val = disease_importance_matrix[i, j]
            text_color = 'white' if val > disease_importance_matrix[~np.isnan(disease_importance_matrix)].max() * 0.6 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=6.5, color=text_color)

plt.colorbar(im, ax=ax, label='Importance (AUROC drop when permuted)', shrink=0.8)
plt.tight_layout()
plt.show()

# ── Print top driver disease for each target ──────────────────────────────────
print("\nTop disease driver for each target:")
print("-" * 65)
for k, target_col in enumerate(disease_cols):
    disease_imps = {col: importance[target_col][col] 
                    for col in disease_cols if col != target_col}
    top_driver = max(disease_imps, key=disease_imps.get)
    top_val = disease_imps[top_driver]
    print(f"  {disease_meta[target_col]:30s} ← {disease_meta[top_driver]:25s} ({top_val:.4f})")

In [ ]:
# ── Full Importance Heatmap: Demographics + Diseases → Disease Targets ────────
import matplotlib.pyplot as plt
import numpy as np

all_input_features = demo_cols + disease_cols
input_labels = [demo_meta.get(col, col) for col in demo_cols] + \
               [disease_meta.get(col, col) for col in disease_cols]
target_labels = [disease_meta[col] for col in disease_cols]

# Build full matrix: rows = targets, cols = all inputs
full_matrix = np.zeros((len(disease_cols), len(all_input_features)))

for k, target_col in enumerate(disease_cols):
    for j, input_col in enumerate(all_input_features):
        if input_col in importance[target_col]:
            full_matrix[k, j] = importance[target_col][input_col]

# Mask diagonal where input == target
for k, target_col in enumerate(disease_cols):
    for j, input_col in enumerate(all_input_features):
        if input_col == target_col:
            full_matrix[k, j] = np.nan

# ── Plot ──────────────────────────────────────────────────────────────────────
n_inputs = len(all_input_features)
n_targets = len(disease_cols)
n_demo = len(demo_cols)

fig, ax = plt.subplots(figsize=(22, 11))

im = ax.imshow(full_matrix, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=np.nanmax(full_matrix))

# Axis ticks
ax.set_xticks(range(n_inputs))
ax.set_yticks(range(n_targets))
ax.set_xticklabels(input_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(target_labels, fontsize=8)

# Vertical divider between demographics and diseases
ax.axvline(x=n_demo - 0.5, color='#1a1a2e', linewidth=2.5, linestyle='--')
ax.text(n_demo / 2 - 0.5, -1.8, 'Demographics', ha='center', fontsize=9,
        fontweight='bold', color='#2c7a4b')
ax.text(n_demo + len(disease_cols) / 2 - 0.5, -1.8, 'Disease flags', ha='center',
        fontsize=9, fontweight='bold', color='#c0392b')

ax.set_xlabel('Input feature (driver)', fontsize=11, labelpad=30)
ax.set_ylabel('Target disease (being predicted)', fontsize=11, labelpad=10)
ax.set_title('Permutation Importance — Demographics and Disease Flags → Disease Targets\n'
             '(AUROC drop when feature is permuted)',
             fontsize=12, fontweight='bold', pad=15)

# Value annotations — only show values > 0.005 to avoid clutter
for i in range(n_targets):
    for j in range(n_inputs):
        val = full_matrix[i, j]
        if not np.isnan(val) and val > 0.005:
            text_color = 'white' if val > np.nanmax(full_matrix) * 0.6 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=6, color=text_color)

plt.colorbar(im, ax=ax, label='Importance (AUROC drop when permuted)', shrink=0.7)
plt.tight_layout()
plt.show()